In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only ".
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Imports

In [2]:
import os
import librosa
import xgboost as xgb
from sklearn.model_selection import train_test_split
from kaggle_secrets import UserSecretsClient
import wandb
from wandb.integration.xgboost import WandbCallback
from sklearn.preprocessing import LabelEncoder

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("login")
os.environ["WANDB_API_KEY"]=wandb_api
wandb.login()
print("Wandb login successful")

wandb: Currently logged in as: 23f1000559 (23f1000559-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb login successful


## Load Dataset

In [4]:
!ls /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00000

bass.wav  drums.wav  other.wav	vocals.wav


In [5]:
genres = sorted(os.listdir("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"))


In [6]:
def stems_to_mashup(song_dir):
    mixed = None
    for stems in os.listdir(song_dir):
        stem_path = os.path.join(song_dir,stems)
        y,sr = librosa.load(stem_path,sr=22050,mono=True)
        if mixed is None:
            mixed = y
        else:
            #match stem length
            min_len = min(len(mixed),len(y))
            mixed = mixed[:min_len]+y[:min_len]
    return mixed
    
def extract_features(y,sr=22050):
    features=[]
    #mfcc-(timbral texture)
    mfcc = librosa.feature.mfcc(y=y,sr=sr,n_mfcc=20)
    features.extend(mfcc.mean(axis=1))
    features.extend(mfcc.std(axis=1))
    #chroma-(pitch/harmony)
    chroma = librosa.feature.chroma_stft(y=y,sr=sr)
    features.extend(chroma.mean(axis=1))
    features.extend(chroma.std(axis=1))
    #rms energy(loudness)
    rms = librosa.feature.rms(y=y)
    features.append(rms.mean())
    features.append(rms.std())
    
    return np.array(features)
    

In [7]:
def MyDataset(root_dir):
    x,y = [],[]
    for genre in os.listdir(root_dir):
        genre_path = os.path.join(root_dir,genre)
        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path,song)
            #mashup audio
            mashup_audio = stems_to_mashup(song_path) 
            #features extraction
            features = extract_features(mashup_audio,)
            x.append(features)
            y.append(genre)
    return np.array(x),np.array(y)
            
    
            

In [8]:
root_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
x,y_raw = MyDataset(root_dir)

In [9]:
# encode labels
le = LabelEncoder()
y= le.fit_transform(y_raw)

In [10]:
#split the data
x_train,x_val,y_train,y_val = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

## Model

In [11]:
# Start a new wandb run 
run = wandb.init(
    # Set the wandb project where this run will be logged.
    project="23f1000559-t12026",
    entity ="23f1000559-indian-institute-of-technology-madras",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.1,
        "n_estimators": 300,
        "max_depth": 6,
        "eval_metric": "mlogloss",
        "random_state": 42,
        "model": "XGBClassifier-v2"
    },
)

wandb: setting up run xg195nm1
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260317_122138-xg195nm1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run usual-lion-20
wandb: ⭐️ View project at https://wandb.ai/23f1000559-indian-institute-of-technology-madras/23f1000559-t12026
wandb: 🚀 View run at https://wandb.ai/23f1000559-indian-institute-of-technology-madras/23f1000559-t12026/runs/xg195nm1


In [12]:
model = xgb.XGBClassifier(
    n_estimators=wandb.config.n_estimators,
    max_depth=wandb.config.max_depth,
    learning_rate=wandb.config.learning_rate,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=wandb.config.random_state,
    callbacks=[WandbCallback(log_model=True)]
)

In [13]:
model.fit(
    x_train,y_train,
    eval_set=[(x_val,y_val)],
    verbose=50
)


[0]	validation_0-mlogloss:2.15088


/usr/local/lib/python3.12/dist-packages/wandb/integration/xgboost/xgboost.py:117: UserWarning: [12:21:41] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  config = model.save_config()


[50]	validation_0-mlogloss:1.03284
[100]	validation_0-mlogloss:0.93922
[150]	validation_0-mlogloss:0.91411
[200]	validation_0-mlogloss:0.91109
[250]	validation_0-mlogloss:0.90992
[299]	validation_0-mlogloss:0.91333


XGBClassifier(base_score=None, booster=None,
              callbacks=[<wandb.integration.xgboost.xgboost.WandbCallback object at 0x7a7966a45c10>],
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [14]:
val_accuracy = model.score(x_val,y_val)
print(f"val accuracy: {val_accuracy:.4f}")
run.log({"val_accuracy":val_accuracy})

val accuracy: 0.7200


## Prediction

In [15]:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

In [16]:
def extract_test(y,sr=22050):
    y,sr= librosa.load(y,sr=sr,mono=True)
    return extract_features(y,sr)

In [17]:
test_features = []
for _ , row in test_df.iterrows():
    path = os.path.join("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup",row["filename"])
    test_features.append(extract_test(path))

x_test = np.array(test_features)
preds = model.predict(x_test)
predicted_genres = le.inverse_transform(preds)

## Submission

In [18]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predicted_genres
})
submission.to_csv("submission.csv", index=False)
print(submission.head())

   id   genre
0   1  hiphop
1   2  reggae
2   3  hiphop
3   4  reggae
4   5  reggae
